In [1]:
import numpy as np

# Load feature
data = np.load("features/features.npz")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(2615, 50, 13)
(641, 50, 13)
(2615, 5)
(641, 5)


In [2]:
import tensorflow as tf

EPOCHS = 100
BATCH_SIZE = 32
num_classes = 5

# flatten
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# dimensione del validation set
val_size = int(0.2 * len(X_train))

# shuffle prima di dividere
train_dataset = train_dataset.shuffle(buffer_size=len(X_train), seed=42)

val_dataset = train_dataset.take(val_size)
train_dataset = train_dataset.skip(val_size)

# batching
train_dataset = train_dataset.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)

# Test MLP
import tf_keras as keras
model = keras.Sequential([
    keras.layers.Input(shape=(650,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(num_classes, activation='softmax')
])

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                20832     
                                                                 
 dense_1 (Dense)             (None, 16)                528       
                                                                 
 dense_2 (Dense)             (None, 8)                 136       
                                                                 
 dense_3 (Dense)             (None, 5)                 45        
                                                                 
Total params: 21541 (84.14 KB)
Trainable params: 21541 (84.14 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [3]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)


Epoch 1/100


66/66 [==============================] - 3s 11ms/step - loss: 1.3254 - accuracy: 0.4512 - val_loss: 0.9595 - val_accuracy: 0.6730
Epoch 2/100
66/66 [==============================] - 0s 5ms/step - loss: 0.7850 - accuracy: 0.7333 - val_loss: 0.6521 - val_accuracy: 0.7801
Epoch 3/100
66/66 [==============================] - 0s 4ms/step - loss: 0.5742 - accuracy: 0.7992 - val_loss: 0.4050 - val_accuracy: 0.8700
Epoch 4/100
66/66 [==============================] - 0s 4ms/step - loss: 0.4561 - accuracy: 0.8504 - val_loss: 0.3853 - val_accuracy: 0.8834
Epoch 5/100
66/66 [==============================] - 0s 4ms/step - loss: 0.3796 - accuracy: 0.8719 - val_loss: 0.3177 - val_accuracy: 0.9044
Epoch 6/100
66/66 [==============================] - 0s 4ms/step - loss: 0.3130 - accuracy: 0.9001 - val_loss: 0.2528 - val_accuracy: 0.9197
Epoch 7/100
66/66 [==============================] - 0s 4ms/step - loss: 0.2757 - accuracy: 0.9025 - val_loss: 0.2116 - val_accuracy: 0.9273
Epoch 8/1

In [4]:
# Valutazione
test_loss, test_acc = model.evaluate(test_dataset)

print("Test accuracy:", test_acc)

21/21 [==============================] - 0s 3ms/step - loss: 1.4480 - accuracy: 0.8019
Test accuracy: 0.8018720746040344


In [13]:
# Only to test QAT
model.save_weights("models/QAware_MLP_weights.h5")

In [14]:
# Quantization Aware Training (QAT)
import tensorflow_model_optimization as tfmot
import tf_keras as keras

EPOCHS = 10

model_QAware = keras.Sequential([
    keras.layers.Input(shape=(650,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(num_classes, activation='softmax')
])

model_QAware.load_weights("models/QAware_MLP_weights.h5")

quantize_model = tfmot.quantization.keras.quantize_model

# q_aware stands for for quantization aware.
q_aware_model = quantize_model(model_QAware)

# `quantize_model` requires a recompile.
q_aware_model.compile(optimizer='adam',
              loss=keras.losses.CategoricalCrossentropy(),
              metrics=['accuracy'])

q_aware_model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer_4 (Quantize  (None, 650)               3         
 Layer)                                                          
                                                                 
 quant_dense_20 (QuantizeWr  (None, 32)                20837     
 apperV2)                                                        
                                                                 
 quant_dense_21 (QuantizeWr  (None, 16)                533       
 apperV2)                                                        
                                                                 
 quant_dense_22 (QuantizeWr  (None, 8)                 141       
 apperV2)                                                        
                                                                 
 quant_dense_23 (QuantizeWr  (None, 5)                

In [15]:
q_aware_model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS)

Epoch 1/10


66/66 [==============================] - 2s 9ms/step - loss: 0.1172 - accuracy: 0.9622 - val_loss: 0.0400 - val_accuracy: 0.9866
Epoch 2/10
66/66 [==============================] - 0s 4ms/step - loss: 0.0293 - accuracy: 0.9943 - val_loss: 0.0215 - val_accuracy: 0.9924
Epoch 3/10
66/66 [==============================] - 0s 4ms/step - loss: 0.0159 - accuracy: 0.9962 - val_loss: 0.0081 - val_accuracy: 0.9981
Epoch 4/10
66/66 [==============================] - 0s 4ms/step - loss: 0.0097 - accuracy: 0.9957 - val_loss: 0.0073 - val_accuracy: 0.9981
Epoch 5/10
66/66 [==============================] - 0s 5ms/step - loss: 0.0076 - accuracy: 0.9971 - val_loss: 0.0035 - val_accuracy: 1.0000
Epoch 6/10
66/66 [==============================] - 0s 5ms/step - loss: 0.0063 - accuracy: 0.9971 - val_loss: 0.0045 - val_accuracy: 0.9981
Epoch 7/10
66/66 [==============================] - 0s 5ms/step - loss: 0.0049 - accuracy: 0.9981 - val_loss: 0.0056 - val_accuracy: 0.9981
Epoch 8/10
66/66 [=============

In [16]:
_, q_aware_model_accuracy = q_aware_model.evaluate(test_dataset)

print('Quant test accuracy:', q_aware_model_accuracy)

21/21 [==============================] - 0s 4ms/step - loss: 1.3210 - accuracy: 0.7972
Quant test accuracy: 0.797191858291626
